In [1]:
import os, sys, re, json, hashlib
from pathlib import Path
from datetime import datetime

NORTHSTAR = "regulators-t10-qa"
NOTEBOOK_PATH = "notebooks/qa/40-regulators-t10-qa.ipynb"
TOWER = 10
TOWER_NAME = "Tower of Regulators"
MIN_SCORE = 70

BROWSER_ROOT = Path("/home/phuc/projects/solace-browser")
SRC = BROWSER_ROOT / "src"
WEB = BROWSER_ROOT / "web"
TESTS = BROWSER_ROOT / "tests"

results = []

def record(probe_id, name, passed, detail=""):
    status = "PASS" if passed else "FAIL"
    results.append({"id": probe_id, "name": name, "status": status, "detail": detail})
    print(f"{status}: {name}" + (f" -- {detail}" if detail else ""))

def read_file(path):
    p = Path(path)
    return p.read_text(encoding='utf-8', errors='replace') if p.exists() else ""

print(f"Tower {TOWER}: {TOWER_NAME}")
assert BROWSER_ROOT.exists()

Tower 10: Tower of Regulators


In [2]:
# Cell 1: GDPR & Cookie Consent (Floors 1, 8, 38)

# P1: Cookie consent banner exists in partials-footer
footer = read_file(WEB / "partials-footer.html")
record("T10-P01", "Cookie consent banner exists",
       'cookie-consent-banner' in footer and 'cookie-consent' in footer,
       "partials-footer.html has cookie-consent-banner div")

# P2: Cookie consent has accept and decline buttons
has_accept = 'solace_cookie_consent' in footer and 'accepted' in footer
has_decline = 'declined' in footer
record("T10-P02", "Cookie consent has accept/decline actions",
       has_accept and has_decline,
       "localStorage key solace_cookie_consent with accepted/declined values")

# P3: Cookie consent stored in localStorage (GDPR Article 7 consent)
record("T10-P03", "Cookie consent uses localStorage for persistence",
       "localStorage.getItem('solace_cookie_consent')" in footer or "localStorage.getItem(\"solace_cookie_consent\")" in footer,
       "Consent state persisted locally")

# P4: Cookie consent CSS classes exist in site.css
sitecss = read_file(WEB / "css" / "site.css")
record("T10-P04", "Cookie consent CSS classes in site.css",
       '.cookie-consent' in sitecss and '.cookie-consent__inner' in sitecss,
       "Styled banner with .cookie-consent classes")

PASS: Cookie consent banner exists -- partials-footer.html has cookie-consent-banner div
PASS: Cookie consent has accept/decline actions -- localStorage key solace_cookie_consent with accepted/declined values
PASS: Cookie consent uses localStorage for persistence -- Consent state persisted locally
PASS: Cookie consent CSS classes in site.css -- Styled banner with .cookie-consent classes


In [3]:
# Cell 2: COPPA Age Gate & Child Safety (Floor 36)

# P5: Age gate modal exists on start page
start = read_file(WEB / "start.html")
record("T10-P05", "COPPA age gate modal on start page",
       'age-gate-modal' in start and '13' in start,
       "Age verification modal present in start.html")

# P6: Age gate has yes/no buttons
record("T10-P06", "Age gate has yes/no buttons",
       'age-gate-yes' in start and 'age-gate-no' in start,
       "Yes (13+) and No buttons present")

# P7: Age gate blocked state for under-13
record("T10-P07", "Age gate shows blocked state for under-13",
       'age-gate-blocked' in start,
       "Blocked message displayed when user says No")

# P8: Age gate uses localStorage for persistence
record("T10-P08", "Age gate uses localStorage",
       'solace_age_verified' in start or 'age-gate' in start,
       "Age verification state handled via localStorage or modal logic")

# P9: Age gate CSS in site.css
record("T10-P09", "Age gate CSS classes in site.css",
       '.age-gate-overlay' in sitecss and '.age-gate-card' in sitecss,
       "Age gate styled with dedicated CSS classes")

PASS: COPPA age gate modal on start page -- Age verification modal present in start.html
PASS: Age gate has yes/no buttons -- Yes (13+) and No buttons present
PASS: Age gate shows blocked state for under-13 -- Blocked message displayed when user says No
PASS: Age gate uses localStorage -- Age verification state handled via localStorage or modal logic
PASS: Age gate CSS classes in site.css -- Age gate styled with dedicated CSS classes


In [4]:
# Cell 3: CSP, CORS, Security Headers (Floors 38, 48)

# P10: Content-Security-Policy meta tags in HTML pages
html_files = list(WEB.glob("*.html"))
csp_count = 0
for f in html_files:
    content = read_file(f)
    if 'Content-Security-Policy' in content or 'content-security-policy' in content:
        csp_count += 1
record("T10-P10", "CSP meta tags present in HTML pages",
       csp_count >= 5,
       f"{csp_count} HTML pages have CSP meta tags")

# P11: CORS restricted to localhost in server.py
server = read_file(WEB / "server.py")
record("T10-P11", "CORS restricted to localhost origins",
       'allowed_origins' in server and 'localhost' in server and '*' not in server.split('allowed_origins')[1].split('\n')[0] if 'allowed_origins' in server else False,
       "server.py has allowed_origins whitelist (no wildcard)")

# P12: SECURITY.md exists at project root
record("T10-P12", "SECURITY.md exists",
       (BROWSER_ROOT / "SECURITY.md").exists(),
       "Security policy document present")

# P13: No GA4/Google Analytics tracking (privacy compliance)
all_html = ''
for f in html_files:
    all_html += read_file(f)
has_ga4 = 'gtag' in all_html.lower() or 'google-analytics' in all_html.lower() or 'googletagmanager' in all_html.lower()
record("T10-P13", "No GA4/Google Analytics tracking in HTML pages",
       not has_ga4,
       "No Google Analytics scripts found in HTML pages")

PASS: CSP meta tags present in HTML pages -- 17 HTML pages have CSP meta tags
PASS: CORS restricted to localhost origins -- server.py has allowed_origins whitelist (no wildcard)
PASS: SECURITY.md exists -- Security policy document present
PASS: No GA4/Google Analytics tracking in HTML pages -- No Google Analytics scripts found in HTML pages


In [5]:
# Cell 4: Privacy & Legal Documentation (Floors 1-8, 42)

# P14: Privacy link in footer
record("T10-P14", "Privacy link in footer",
       'privacy' in footer.lower() or 'Privacy' in footer,
       "Footer contains privacy-related link")

# P15: Terms/Legal link in footer
record("T10-P15", "Terms or legal link in footer",
       'terms' in footer.lower() or 'legal' in footer.lower() or 'Terms' in footer,
       "Footer contains terms/legal link")

# P16: LICENSE file exists with FSL
license_content = read_file(BROWSER_ROOT / "LICENSE")
record("T10-P16", "LICENSE file exists with FSL-1.1-Apache-2.0",
       'FSL' in license_content and 'Apache' in license_content,
       "Functional Source License present")

# P17: NOTICE file exists with third-party attributions
notice = read_file(BROWSER_ROOT / "NOTICE")
record("T10-P17", "NOTICE file exists with third-party attributions",
       len(notice) > 100 and 'Third-Party' in notice or 'third-party' in notice.lower(),
       "Third-party software notices present")

# P18: Support page exists
support_exists = (WEB / "support.html").exists() or 'support' in footer.lower()
record("T10-P18", "Support page or link exists",
       support_exists,
       "Users have access to support channel")

PASS: Privacy link in footer -- Footer contains privacy-related link
PASS: Terms or legal link in footer -- Footer contains terms/legal link
PASS: LICENSE file exists with FSL-1.1-Apache-2.0 -- Functional Source License present
PASS: NOTICE file exists with third-party attributions -- Third-party software notices present
PASS: Support page or link exists -- Users have access to support channel


In [6]:
# Cell 5: Data Handling & Evidence (Floors 10, 16, 43-44)

# P19: Evidence chain module exists
chain_py = read_file(SRC / "audit" / "chain.py")
record("T10-P19", "Evidence chain module (audit/chain.py) exists",
       'AuditChain' in chain_py or 'EvidenceChainManager' in chain_py,
       "SHA-256 hash chain implementation present")

# P20: Evidence chain uses SHA-256
record("T10-P20", "Evidence chain uses SHA-256 hashing",
       'sha256' in chain_py or 'SHA-256' in chain_py,
       "Tamper-evident hash chain for audit trails")

# P21: OAuth3 scopes module exists for data access control
scopes_py = read_file(SRC / "oauth3" / "scopes.py")
record("T10-P21", "OAuth3 scopes module exists",
       'validate_scopes' in scopes_py,
       "Scope-based access control for data processing")

# P22: OAuth3 consent UI exists
consent_py = read_file(SRC / "oauth3" / "consent_ui.py")
record("T10-P22", "OAuth3 consent UI module exists",
       'build_consent_page' in consent_py,
       "User-facing consent page for data access approvals")

# P23: OAuth3 revocation module exists
revocation_py = read_file(SRC / "oauth3" / "revocation.py")
record("T10-P23", "OAuth3 revocation module exists",
       len(revocation_py) > 50,
       "Token revocation capability for right to withdraw consent")

PASS: Evidence chain module (audit/chain.py) exists -- SHA-256 hash chain implementation present
PASS: Evidence chain uses SHA-256 hashing -- Tamper-evident hash chain for audit trails
PASS: OAuth3 scopes module exists -- Scope-based access control for data processing
PASS: OAuth3 consent UI module exists -- User-facing consent page for data access approvals
PASS: OAuth3 revocation module exists -- Token revocation capability for right to withdraw consent


In [7]:
# Cell 6: Accessibility & Internationalization (Floors 3, 14, 31)

# P24: Locale files exist for multiple languages (LGPD: Portuguese, Health Canada: French)
locale_dir = BROWSER_ROOT / "app" / "locales" / "yinyang"
locale_count = len(list(locale_dir.glob("*.json"))) if locale_dir.exists() else 0
record("T10-P24", "Multiple locale files exist (47 languages)",
       locale_count >= 40,
       f"{locale_count} locale files found")

# P25: Portuguese locale exists (LGPD compliance)
record("T10-P25", "Portuguese locale exists (LGPD)",
       (locale_dir / "pt.json").exists() if locale_dir.exists() else False,
       "pt.json locale file for Brazilian users")

# P26: French locale exists (Health Canada bilingual requirement)
record("T10-P26", "French locale exists (Health Canada)",
       (locale_dir / "fr.json").exists() if locale_dir.exists() else False,
       "fr.json locale file for Canadian bilingual compliance")

# P27: robots.txt exists with proper directives
robots = read_file(WEB / "robots.txt")
record("T10-P27", "robots.txt exists with proper directives",
       'User-agent' in robots and 'Disallow' in robots,
       "Web crawler directives present")

# P28: sitemap.xml exists
sitemap = read_file(WEB / "sitemap.xml")
record("T10-P28", "sitemap.xml exists",
       'urlset' in sitemap and 'loc' in sitemap,
       "XML sitemap for search engine discovery")

PASS: Multiple locale files exist (47 languages) -- 47 locale files found
PASS: Portuguese locale exists (LGPD) -- pt.json locale file for Brazilian users
PASS: French locale exists (Health Canada) -- fr.json locale file for Canadian bilingual compliance
PASS: robots.txt exists with proper directives -- Web crawler directives present
PASS: sitemap.xml exists -- XML sitemap for search engine discovery


In [8]:
# Evidence Summary Cell
passed = len([r for r in results if r["status"] == "PASS"])
failed = len([r for r in results if r["status"] == "FAIL"])
total = len(results)
score = round(passed / total * 100, 1) if total > 0 else 0
finding_rate = round(failed / total * 100, 1) if total > 0 else 0
grade = "A+" if score >= 95 else "A" if score >= 90 else "B+" if score >= 85 else "B" if score >= 80 else "C" if score >= 70 else "D" if score >= 60 else "F"
summary = {
    "surface": NORTHSTAR, "tower": TOWER, "tower_name": TOWER_NAME,
    "timestamp": datetime.now().isoformat(),
    "total_probes": total, "passed": passed, "failed": failed,
    "score": score, "grade": grade, "finding_rate": finding_rate,
    "converged": finding_rate < 5,
    "findings": [r for r in results if r["status"] == "FAIL"],
    "evidence_hash": hashlib.sha256(json.dumps(results, sort_keys=True).encode()).hexdigest()[:16]
}
print("=" * 60)
print(f"TOWER {TOWER}: {TOWER_NAME}")
print("=" * 60)
print(f"Probes: {total} | Passed: {passed} | Failed: {failed}")
print(f"Score: {score}% | Grade: {grade} | Finding rate: {finding_rate}%")
print(f"Converged: {'YES' if summary['converged'] else 'NO'}")
print(f"Evidence hash: {summary['evidence_hash']}")
if summary["findings"]:
    print("\nFINDINGS:")
    for f in summary["findings"]:
        print(f"  [{f['id']}] {f['name']}: {f.get('detail', '')}")
print()
print(json.dumps(summary, indent=2))

TOWER 10: Tower of Regulators
Probes: 28 | Passed: 28 | Failed: 0
Score: 100.0% | Grade: A+ | Finding rate: 0.0%
Converged: YES
Evidence hash: 151381390541c3fb

{
  "surface": "regulators-t10-qa",
  "tower": 10,
  "tower_name": "Tower of Regulators",
  "timestamp": "2026-03-06T11:30:16.530303",
  "total_probes": 28,
  "passed": 28,
  "failed": 0,
  "score": 100.0,
  "grade": "A+",
  "finding_rate": 0.0,
  "converged": true,
  "findings": [],
  "evidence_hash": "151381390541c3fb"
}
